In [12]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from lifelines.statistics import logrank_test
from lifelines import CoxPHFitter
import pickle
import warnings

warnings.filterwarnings("ignore")
random_state = 42

# Dictionary to store intermediate results
results_tracker = {}

def load_data(base_path):
    """Load clinical and methylation data safely."""
    file_paths = {
        "clinical_data": os.path.join(base_path, "2019_TCGA-CDR-SupplementalTableS1.xlsx"),
        "new_labels": os.path.join(base_path, "Matrix_WHO2021.csv"),
        "gbm_data": os.path.join(base_path, "GBM_450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"),
        "lgg_data": os.path.join(base_path, "LGG-450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"),
    }

    # Check if all files exist
    for key, path in file_paths.items():
        if not os.path.exists(path):
            print(f"❌ Error: Missing {key} file: {path}")
            return None

    # Load data
    print("📂 Loading data...")
    main_data_df = pd.read_excel(file_paths["clinical_data"])
    new_categories = pd.read_csv(file_paths["new_labels"])
    gbm_data = pd.read_csv(file_paths["gbm_data"])
    lgg_data = pd.read_csv(file_paths["lgg_data"])

    print("✅ Data Loaded Successfully!")
    results_tracker["data_load"] = {
        "clinical_data": main_data_df.shape,
        "new_labels": new_categories.shape,
        "gbm_data": gbm_data.shape,
        "lgg_data": lgg_data.shape
    }
    return main_data_df, new_categories, gbm_data, lgg_data

def preprocess_methylation_data(df):
    """Transpose and clean methylation data."""
    df_trns = df.T
    df_trns.columns = df_trns.iloc[0]
    return df_trns[1:]

def merge_data(main_data_df, new_categories, gbm_data, lgg_data):
    """Merge clinical and methylation data."""
    main_data_df.rename(columns={'bcr_patient_barcode': 'Patient_ID'}, inplace=True)
    filtered_data_df = main_data_df[main_data_df['type'].isin(['GBM', 'LGG'])]
    merged_clinical_data = pd.merge(filtered_data_df, new_categories, on='Patient_ID', how='inner')

    gbm_trns_final = preprocess_methylation_data(gbm_data)
    lgg_trns_final = preprocess_methylation_data(lgg_data)
    merged_methylation_data = pd.concat([gbm_trns_final, lgg_trns_final])
    merged_methylation_data.rename(columns={'Index': 'Patient_ID'}, inplace=True)

    final_data = pd.merge(merged_methylation_data, merged_clinical_data, left_index=True, right_on='Patient_ID', how='inner')

    print(f"✅ Final Merged Data Shape: {final_data.shape}")
    results_tracker["merged_data"] = final_data.shape
    return final_data

def split_data(df):
    """Split data into three subsets based on classification labels."""
    df = df[df['classification.2021_simplified.labels'] != 'unclassified']
    
    subsets = {
        'astrocytoma': df[df['classification.2021_simplified.labels'] == 'astrocytoma'],
        'glioblastoma': df[df['classification.2021_simplified.labels'] == 'glioblastoma'],
        'oligodendroglioma': df[df['classification.2021_simplified.labels'] == 'oligodendroglioma']
    }
    
    print(f"✅ Data Split into: {', '.join(subsets.keys())}")
    results_tracker["split_data"] = {k: v.shape for k, v in subsets.items()}
    return subsets





In [13]:
def feature_selection(df, label):
    """Perform feature selection and survival analysis with NaN handling."""

    print(f"\n🔹 Processing {label} Data...")

    # Drop NaNs only in target variables (`OS.time`, `OS`) to retain other data
    df = df.dropna(subset=["OS.time", "OS"])

    # Check if there are enough samples after removing NaNs
    if df.shape[0] < 2:
        print(f"❌ Not enough valid samples for {label} after removing NaNs in `OS.time`. Skipping...")
        return None

    print(f"✅ Data Shape After Target NaN Removal: {df.shape}")

    # Convert to numeric, ensuring non-numeric values are set to NaN
    df = df.apply(pd.to_numeric, errors="coerce")

    # Select only methylation feature columns
    methylation_features = [col for col in df.columns if col.startswith("cg")]

    if len(methylation_features) == 0:
        print(f"❌ No valid methylation features found for {label}. Skipping.")
        return None

    X = df[methylation_features]
    y = df[['OS.time', 'OS']]

    # Ensure enough samples for train-test split
    if X.shape[0] < 2:
        print(f"❌ Not enough samples for {label} to perform train-test split. Skipping...")
        return None

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

    print(f"✅ {label} Train-Test Split: Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

    # Variance thresholding (removes near-constant features)
    variance_thresh = VarianceThreshold(threshold=0.01)

    try:
        X_train_reduced = variance_thresh.fit_transform(X_train)
        selected_features = X_train.columns[variance_thresh.get_support()]
        X_test_reduced = X_test[selected_features]
    except ValueError:
        print(f"❌ Variance thresholding failed for {label}. Skipping.")
        return None

    if len(selected_features) == 0:
        print(f"❌ No features left after variance thresholding for {label}. Skipping.")
        return None

    # Survival analysis with log-rank test
    feature_pvalues = {}
    X_train_reduced_df = pd.DataFrame(X_train_reduced, columns=selected_features, index=X_train.index)

    for feature in selected_features:
        median_value = X_train_reduced_df[feature].median()
        high_group = y_train[X_train_reduced_df[feature] >= median_value]
        low_group = y_train[X_train_reduced_df[feature] < median_value]

        if high_group.shape[0] < 2 or low_group.shape[0] < 2:
            continue

        result = logrank_test(high_group["OS.time"], low_group["OS.time"], 
                              event_observed_A=high_group["OS"], event_observed_B=low_group["OS"])
        feature_pvalues[feature] = result.p_value

    if not feature_pvalues:
        print(f"❌ No valid p-values computed for {label}. Skipping.")
        return None

    pval_df = pd.DataFrame.from_dict(feature_pvalues, orient="index", columns=["p_value"])
    pval_df.sort_values("p_value", inplace=True)

    # Select significant features (p-value < 0.0001)
    significant_features = pval_df[pval_df["p_value"] < 0.0001].index.tolist()

    if len(significant_features) == 0:
        print(f"❌ No significant features found for {label}. Skipping.")
        return None

    # Remove highly correlated features
    corr_matrix = X_train_reduced_df[significant_features].corr().abs()
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.4)]  # Stricter threshold

    final_selected_features = list(set(significant_features) - set(to_drop))

    if len(final_selected_features) == 0:
        print(f"❌ No features left after correlation filtering for {label}. Skipping.")
        return None

    # Cox Proportional Hazards Model
    X_train_cox = X_train_reduced_df[final_selected_features]
    df_train_cox = X_train_cox.copy()
    df_train_cox["OS.time"] = y_train["OS.time"]
    df_train_cox["OS"] = y_train["OS"].astype(bool)

    try:
        cox_model = CoxPHFitter()
        cox_model.fit(df_train_cox, duration_col="OS.time", event_col="OS")

        # Compute train and test C-index manually
        from lifelines.utils import concordance_index

        train_c_index = concordance_index(df_train_cox["OS.time"], -cox_model.predict_partial_hazard(df_train_cox), df_train_cox["OS"])

        df_test_cox = pd.DataFrame(X_test[final_selected_features], columns=final_selected_features)
        df_test_cox["OS.time"] = y_test["OS.time"]
        df_test_cox["OS"] = y_test["OS"].astype(bool)

        test_c_index = concordance_index(df_test_cox["OS.time"], -cox_model.predict_partial_hazard(df_test_cox), df_test_cox["OS"])

        cox_summary = cox_model.summary.sort_values(by="exp(coef)", ascending=False)
        significant_cox_features = cox_summary[cox_summary["p"] < 0.05].index.tolist()

    except Exception as e:
        print(f"❌ Cox model fitting failed for {label}: {e}")
        return None

    # Save results
    results_tracker[label] = {
        "train_c_index": train_c_index,
        "test_c_index": test_c_index,
        "significant_features": significant_cox_features,
        "cox_summary": cox_summary
    }

    print(f"✅ {label} Processed. Train C-Index: {train_c_index:.4f} | Test C-Index: {test_c_index:.4f}")

    return {
        "X_train": X_train[significant_cox_features],
        "X_test": X_test[significant_cox_features],
        "y_train": y_train,
        "y_test": y_test,
        "cox_summary": cox_summary,
        "train_c_index": train_c_index,
        "test_c_index": test_c_index
    }


In [ ]:

base_path = "/Users/nijat/Downloads/OneDrive_1_2024-09-21/Data"

merged_data = merge_data(*load_data(base_path))
subsets = split_data(merged_data)

for label, subset_df in subsets.items():
    print(f"\n🔹 Processing {label} Data...")

    results = feature_selection(subset_df, label)

    if results is None:
        print(f"❌ No valid results for {label}. Skipping.")
        continue


    results_tracker[label] = results

    results_file = f"glioma_results_{label}.pkl"
    with open(results_file, "wb") as f:
        pickle.dump(results, f)
    print(f"📁 Results saved: {results_file}")

    selected_probes_file = f"selected_probes_{label}.csv"
    selected_probes_df = pd.DataFrame(results["X_train"].columns, columns=["Selected_Probes"])
    selected_probes_df.to_csv(selected_probes_file, index=False)
    print(f"📁 Selected probes saved: {selected_probes_file}")

    print(f"🔍 Selected Probes for {label}: {list(results['X_train'].columns)}")
    print(f"✅ {label} Concordance Index (C-index) → Train: {results['train_c_index']:.4f} | Test: {results['test_c_index']:.4f}")

# Save all results in a master pickle file
with open("glioma_results_all.pkl", "wb") as f:
    pickle.dump(results_tracker, f)

print("\n✅ All Processing Completed. Results Saved in 'glioma_results_all.pkl'.")


📂 Loading data...
✅ Data Loaded Successfully!
✅ Final Merged Data Shape: (653, 403991)
✅ Data Split into: astrocytoma, glioblastoma, oligodendroglioma

🔹 Processing astrocytoma Data...

🔹 Processing astrocytoma Data...
✅ Data Shape After Target NaN Removal: (256, 403991)
✅ astrocytoma Train-Test Split: Train: 204, Test: 52
✅ astrocytoma Processed. Train C-Index: 0.8779 | Test C-Index: 0.6682
📁 Results saved: glioma_results_astrocytoma.pkl
📁 Selected probes saved: selected_probes_astrocytoma.csv
🔍 Selected Probes for astrocytoma: ['cg08938584', 'cg06690831', 'cg14199570']
✅ astrocytoma Concordance Index (C-index) → Train: 0.8779 | Test: 0.6682

🔹 Processing glioblastoma Data...

🔹 Processing glioblastoma Data...
✅ Data Shape After Target NaN Removal: (194, 403991)
✅ glioblastoma Train-Test Split: Train: 155, Test: 39
✅ glioblastoma Processed. Train C-Index: 0.7143 | Test C-Index: 0.4554
📁 Results saved: glioma_results_glioblastoma.pkl
📁 Selected probes saved: selected_probes_glioblastom